In [2]:
import pandas as pd
import numpy as np


In [3]:
data = pd.read_csv('../data/clean_data.csv',parse_dates=['datetime'])
print(f'''Список отслеживаемых событий: 
{data['EventName'].unique()}''')

Список отслеживаемых событий: 
['Tutorial' 'MainScreenAppear' 'OffersScreenAppear' 'CartScreenAppear'
 'PaymentScreenSuccessful']


In [4]:
# отсортируем по пользователю и времени заранее, чтобы не забыть.

data = data.sort_values(by =['DeviceIDHash','datetime'])
#data.head(3)
data.info()
data.head(3)

<class 'pandas.core.frame.DataFrame'>
Index: 241298 entries, 194435 to 218610
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   EventName     241298 non-null  object        
 1   DeviceIDHash  241298 non-null  int64         
 2   ExpId         241298 non-null  int64         
 3   datetime      241298 non-null  datetime64[ns]
 4   date          241298 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 11.0+ MB


,EventName,DeviceIDHash,ExpId,datetime,date
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06
206368,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06
206371,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06


In [5]:
# Количество событий без учета групп и количество уникальных пользовтелей по каждому событию
print('События - пользователи')
event_users = (data.groupby('EventName')
               .agg(cnt_event = ('EventName', 'count'), 
                    cnt_uniq_users = ('DeviceIDHash', 'nunique')
                    )
               .sort_values('cnt_event',ascending=False)
               .reset_index()
)
print(event_users)

# А сколько всего уникальных пользователей
print(f''' 
Уникальных пользователей: {data['DeviceIDHash'].nunique()}''')

# Уникальных > Зашедших на стартовую страницу!

# Доля пользователей, совершивших событие
ratio_event_users = event_users[['EventName', 'cnt_uniq_users']]
ratio_event_users['ratio'] = round(ratio_event_users['cnt_uniq_users'] / data['DeviceIDHash'].nunique() * 100, 2)
print(ratio_event_users)


События - пользователи
                 EventName  cnt_event  cnt_uniq_users
0         MainScreenAppear     117431            7419
1       OffersScreenAppear      46350            4593
2         CartScreenAppear      42365            3734
3  PaymentScreenSuccessful      34113            3539
4                 Tutorial       1039             840
 
Уникальных пользователей: 7534
                 EventName  cnt_uniq_users  ratio
0         MainScreenAppear            7419  98.47
1       OffersScreenAppear            4593  60.96
2         CartScreenAppear            3734  49.56
3  PaymentScreenSuccessful            3539  46.97
4                 Tutorial             840  11.15


In [6]:
# Почему количество уникальных не совпадает с количество уникальных, видивших стартовый экран приложения

# перепроверим 
main_users = set(data[data['EventName'] == 'MainScreenAppear']['DeviceIDHash']) # уникальные пользователи, имеющие события MainScreenAppear
all_users = set(data['DeviceIDHash']) # все уникальные пользователи
not_main_users = all_users - main_users
print(f'Пользователи, которые не видели MainScreenAppear: {len(not_main_users)}')

# фильтруем пользователей "без основного экрана"
not_main_events = data[data['DeviceIDHash'].isin(not_main_users)]

# какие события встречаются у данных пользователей
not_main_users_events = not_main_events.groupby('DeviceIDHash').agg(
    event_list=('EventName', list),
    expid=('ExpId', 'first'),
    events_count=('EventName', 'count')  
).reset_index()

print("Пользователи, не дошедшие до MainScreenAppear:")
print(not_main_users_events)


# пользователи без покупок
not_payment = not_main_users_events[~not_main_users_events['event_list'].apply(lambda x: 'PaymentScreenSuccessful' in x)]
print(f'Пришедших извне и не сделавших покупок {len(not_payment)}')
not_payment

# вывод: не все пользователи заходят в приложение через ярлык. Есть другие способы (пуш, ссылка от партнеров, смс)

Пользователи, которые не видели MainScreenAppear: 115
Пользователи, не дошедшие до MainScreenAppear:
            DeviceIDHash                                         event_list  \
0      74158328448226259  [PaymentScreenSuccessful, CartScreenAppear, Of...   
1     111394506613435756  [PaymentScreenSuccessful, CartScreenAppear, Of...   
2     214966247576341063  [OffersScreenAppear, PaymentScreenSuccessful, ...   
3     261817378841141406  [PaymentScreenSuccessful, CartScreenAppear, Of...   
4     332529825412858125  [OffersScreenAppear, CartScreenAppear, OffersS...   
..                   ...                                                ...   
110  8586953157808767383  [OffersScreenAppear, CartScreenAppear, Payment...   
111  8804319115517716344  [PaymentScreenSuccessful, CartScreenAppear, Of...   
112  8821171531680573201  [OffersScreenAppear, PaymentScreenSuccessful, ...   
113  9124766629178994679  [OffersScreenAppear, OffersScreenAppear, Offer...   
114  9217594193087726423  [Pay

,DeviceIDHash,event_list,expid,events_count
12,1223708690315846789,[OffersScreenAppear],246,1
15,1478347681767261393,"[OffersScreenAppear, OffersScreenAppear, Offer...",247,4
19,1900791869709139147,[OffersScreenAppear],248,1
20,1958496982439584534,[OffersScreenAppear],248,1
25,2472435690120708424,[OffersScreenAppear],246,1
27,2525867977967066505,"[Tutorial, Tutorial, Tutorial]",247,3
34,2974131200515436842,"[OffersScreenAppear, OffersScreenAppear]",248,2
43,3800345857856674685,"[Tutorial, Tutorial]",248,2
48,4111480625662873403,"[OffersScreenAppear, CartScreenAppear, CartScr...",247,3
57,4530866328707864508,[Tutorial],248,1


In [7]:
# пришедших извне в каждой группе А/А/В
print(not_main_users_events.groupby('expid')['DeviceIDHash'].nunique())

expid
246    34
247    37
248    44
Name: DeviceIDHash, dtype: int64


In [8]:
# малоактивные пользователи
counts = data.groupby('DeviceIDHash')['EventName'].transform('count')
one_events = data[counts <= 1]
#print(one_events)
print(' Уникальных неактивных пользователей')
print(one_events['DeviceIDHash'].nunique())
print(f' С разбивкой по группам {one_events.groupby('ExpId')['DeviceIDHash'].nunique()}')
one_events.groupby('EventName').agg(users = ('DeviceIDHash', 'nunique'), cnt_event = ('EventName', 'count'))


 Уникальных неактивных пользователей
119
 С разбивкой по группам ExpId
246    36
247    40
248    43
Name: DeviceIDHash, dtype: int64


,users,cnt_event
EventName,,
MainScreenAppear,111,111
OffersScreenAppear,6,6
Tutorial,2,2


In [5]:
#data.groupby(['ExpId', 'EventName']).agg(cnt_event = ('EventName', 'count'), )

cnt_event
ExpId EventName                         
246   CartScreenAppear             14711
      MainScreenAppear             37708
      OffersScreenAppear           14773
      PaymentScreenSuccessful      11910
      Tutorial                       323
247   CartScreenAppear             12456
      MainScreenAppear             39123
      OffersScreenAppear           15182
      PaymentScreenSuccessful      10043
      Tutorial                       343
248   CartScreenAppear             15198
      MainScreenAppear             40600
      OffersScreenAppear           16395
      PaymentScreenSuccessful      12160
      Tutorial                       373

In [33]:
# Анализ последовательности событий на примере пользователя    214966247576341063 6922444491712477
data.query('DeviceIDHash == 9222603179720523844')

,EventName,DeviceIDHash,ExpId,datetime,date,session
4593,MainScreenAppear,9222603179720523844,248,2019-08-01 06:52:13,2019-08-01,1
4610,MainScreenAppear,9222603179720523844,248,2019-08-01 06:52:56,2019-08-01,1
4628,MainScreenAppear,9222603179720523844,248,2019-08-01 06:53:30,2019-08-01,1
4666,MainScreenAppear,9222603179720523844,248,2019-08-01 06:54:26,2019-08-01,1
4673,MainScreenAppear,9222603179720523844,248,2019-08-01 06:54:35,2019-08-01,1
4701,MainScreenAppear,9222603179720523844,248,2019-08-01 06:55:59,2019-08-01,1
32829,MainScreenAppear,9222603179720523844,248,2019-08-01 19:17:59,2019-08-01,2
32855,MainScreenAppear,9222603179720523844,248,2019-08-01 19:19:15,2019-08-01,2
32904,MainScreenAppear,9222603179720523844,248,2019-08-01 19:20:24,2019-08-01,2
32910,MainScreenAppear,9222603179720523844,248,2019-08-01 19:20:35,2019-08-01,2


In [106]:
# пробуем выделить сессии  - неправильный метод, ибо сравнивает с уже имеющейся колонкой, а в ней все нули. А очень хотелось использовать np.select )) Попробовать позже инач сделать

#data1 = data.copy()
#data1['session'] = 0
#limit = pd.Timedelta(minutes = 30)
#conditions = [
#    data1['DeviceIDHash'] != data1.groupby('DeviceIDHash')['DeviceIDHash'].shift(-1),
#    data1['datetime'] <= data1.groupby('DeviceIDHash')['datetime'].shift(-1) + limit,
#    data1['datetime'] > data1.groupby('DeviceIDHash')['datetime'].shift(-1) + limit
#]
#choices = [
#    1,
#    data1.groupby('DeviceIDHash')['session'].shift(-1),
#    data1.groupby('DeviceIDHash')['DeviceIDHash'].shift(-1) + 1
#]
#
#data1['session'] = np.select(conditions,choices)
#data1

,EventName,DeviceIDHash,ExpId,datetime,date,session
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06,1.0
206368,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06,0.0
206371,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
206372,CartScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
206373,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
...,...,...,...,...,...,...
218538,MainScreenAppear,9222603179720523844,248,2019-08-07 09:13:37,2019-08-07,0.0
218576,MainScreenAppear,9222603179720523844,248,2019-08-07 09:14:53,2019-08-07,0.0
218578,MainScreenAppear,9222603179720523844,248,2019-08-07 09:15:01,2019-08-07,0.0
218584,MainScreenAppear,9222603179720523844,248,2019-08-07 09:15:13,2019-08-07,0.0


In [9]:
limit = pd.Timedelta(minutes=30) # зададим лимит на сессию

new_user = data['DeviceIDHash'] != data['DeviceIDHash'].shift(1) # помечаем где начинается новый пользователь, с него будет новый счетчик сессий
time_diff = data.groupby('DeviceIDHash')['datetime'].diff() # смотрим разницу по времени с предыдущим событием
timeout = time_diff > limit # помечаем где разница больше лимита на сессию
new_session = new_user | timeout.fillna(True) # новая сессия = либо новый пользователь либо превышен лимит

# Нумеруем сессии ВНУТРИ каждого пользователя
data['session'] = new_session.groupby(data['DeviceIDHash']).cumsum()

# Общее количество сессий на период наблюдений 
print(data.groupby('DeviceIDHash')['session'].max().sum())

In [16]:
data.head(30)

,EventName,DeviceIDHash,ExpId,datetime,date,session
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06,1
206368,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06,1
206371,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1
206372,CartScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1
206373,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,1
206382,OffersScreenAppear,6909561520679493,247,2019-08-06 18:53:04,2019-08-06,1
124842,MainScreenAppear,6922444491712477,246,2019-08-04 14:19:33,2019-08-04,1
124844,PaymentScreenSuccessful,6922444491712477,246,2019-08-04 14:19:40,2019-08-04,1
124845,CartScreenAppear,6922444491712477,246,2019-08-04 14:19:40,2019-08-04,1
124846,MainScreenAppear,6922444491712477,246,2019-08-04 14:19:40,2019-08-04,1


In [31]:
data.groupby(['ExpId', 'DeviceIDHash', 'session']).agg(time_session = ('datetime', lambda x: (max(x) - min(x)).total_seconds()),
                                              PaySuccess = ('EventName', lambda x: 'PaymentScreenSuccessful' in list(x)),
                                              cnt_pay_session = ('EventName', lambda x: list(x).count('PaymentScreenSuccessful')),
                                              total_actions = ('EventName', lambda x: len(x)),
                                              viewOffers = ('EventName', lambda x: 'OffersScreenAppear' in list(x)),
                                              from_outside = ('EventName', lambda x: 'MainScreenAppear' != x.iloc[0])

)

# посмотреть те сессии, где время = 0 (одна активность за сессию). Если такое действие встречается часто и у активных пользователей, 
# то может означать, что часто вылетает программа при входе                                             

time_session  PaySuccess  cnt_pay_session  \
ExpId DeviceIDHash        session                                              
246   6888746892508752    1                 0.0       False                0   
      6922444491712477    1                13.0        True                1   
                          2                16.0        True                1   
                          3               335.0       False                0   
                          4               925.0        True                5   
...                                         ...         ...              ...   
248   9222603179720523844 7                 0.0       False                0   
                          8              1628.0       False                0   
                          9               840.0       False                0   
                          10              624.0       False                0   
                          11              283.0       False                0   

                                   total_actions  viewOffers  from_outside  
ExpId DeviceIDHash        session                                           
246   6888746892508752    1                    1       False         False  
      6922444491712477    1                    5        True         False  
                          2                    5        True         False  
                          3                    6        True         False  
                          4                   17        True         False  
...                                          ...         ...           ...  
248   9222603179720523844 7                    1       False         False  
                          8                   10       False         False  
                          9                    6       False         False  
                          10                   2       False         False  
                          11                   6       False         False  

[45107 rows x 6 columns]

In [42]:
# Общее количество сессий на период наблюдений 
#print(data.groupby('DeviceIDHash')['session'].max().sum())



array([9198403615978777375, 9200038491366259406, 9210082518034880206,
       9211196294399965012, 9212377414136562136, 9212420551954885212,
       9212523802225607780, 9214668690707156694, 9219463515465815368,
       9222603179720523844])

In [ ]:
# нужно сравнить группы А и А - равны ли их основные характеристики, чтобы потом сравнить их с группой В

# пользователь 9222603179720523844 имеет только событие MainScreenAppear и больше ничего!, хотя время проводит много в приложении. Кто он, сколько таких? 
# Ошибка приложения, тестовый пользователь 